# Cross-Encoder Reranker Pipeline — Notebook Version

| Phase | Mô tả |
|-------|-------|
| **A** | EDA & Sanity Check |
| **B** | Tạo `eval_qa.jsonl` |
| **C** | Train Cross-Encoder |
| **D** | Evaluate: Classification + Ranking + Reranking |
| **E** | End-to-End Pipeline (Retrieve → Rerank → Gate → LLM) |
| **F** | Tạo DELIVERABLES.md |

> Chạy từng cell theo thứ tự, hoặc **Run All** nếu muốn chạy toàn bộ.

## Cell 0 — Config & Imports

In [1]:
import json, os, sys, time, re, csv, random, shutil
from datetime import datetime
from pathlib import Path
from collections import Counter

# ── Paths ──
ROOT = Path(".")
DATA_DIR = ROOT / "data"
OUT_DIR  = ROOT / "outputs"
EDA_DIR  = OUT_DIR / "eda"
EVAL_DIR = OUT_DIR / "eval"
MDL_DIR  = OUT_DIR / "models"
TMP_DIR  = OUT_DIR / "tmp"
LOG_DIR  = OUT_DIR / "logs"

TRAIN_FILE    = DATA_DIR / "train.jsonl"
DEV_FILE      = DATA_DIR / "dev.jsonl"
TRAIN_NEG     = DATA_DIR / "train_with_neg.jsonl"
MODEL_OUT     = MDL_DIR  / "cross_encoder_v1"
EVAL_QA_FILE  = EVAL_DIR / "eval_qa.jsonl"
DEV_NEG_FILE  = EVAL_DIR / "dev_with_neg.jsonl"
FAISS_INDEX   = TMP_DIR  / "faiss.index"
FAISS_MAP     = TMP_DIR  / "faiss_mapping.jsonl"
CLASS_METRICS = EVAL_DIR / "dev_classification_metrics.json"
RERANK_CSV    = EVAL_DIR / "rerank_metrics.csv"
PIPE_RESULTS  = EVAL_DIR / "pipeline_results.jsonl"
PIPE_SUM_CSV  = EVAL_DIR / "pipeline_summary.csv"
PIPE_SUM_MD   = EVAL_DIR / "pipeline_summary.md"
DELIVERABLES  = OUT_DIR  / "DELIVERABLES.md"

# ── Model config ──
BASE_CE_MODEL = "cross-encoder/ms-marco-MiniLM-L-6-v2"
BASE_BI_MODEL = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
SEED       = 42
EPOCHS     = 3
BATCH_SIZE = 32
MAX_LENGTH = 256
TOP_N      = 50
EVAL_N     = 50

CJK_RE = re.compile(r'[\u4e00-\u9fff\u3040-\u309f\u30a0-\u30ff]')

for d in [EDA_DIR/"plots", EVAL_DIR, MDL_DIR, TMP_DIR, LOG_DIR]:
    d.mkdir(parents=True, exist_ok=True)

def log(msg):
    print(f"[{datetime.now().strftime('%H:%M:%S')}] {msg}", flush=True)

def load_jsonl(path, max_rows=None):
    rows, errors = [], 0
    with open(path, encoding="utf-8", errors="replace") as f:
        for i, line in enumerate(f):
            if max_rows and i >= max_rows: break
            line = line.strip()
            if not line: continue
            try: rows.append(json.loads(line))
            except json.JSONDecodeError: errors += 1
    if errors: log(f"  ⚠ {errors} malformed lines skipped in {Path(path).name}")
    return rows

def write_jsonl(path, rows):
    with open(path, "w", encoding="utf-8") as f:
        for r in rows: f.write(json.dumps(r, ensure_ascii=False) + "\n")

def count_cjk(text): return len(CJK_RE.findall(text))

def avg(lst): return round(sum(lst)/len(lst), 4) if lst else 0.0

import torch
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
random.seed(SEED)
log(f"torch={torch.__version__} | device={DEVICE} | seed={SEED}")
print("Config loaded ✓")

[03:06:23] torch=2.6.0+cu124 | device=cuda | seed=42
Config loaded ✓


---
## Phase A — EDA & Sanity Check

In [2]:
log("=" * 60)
log("PHASE A: EDA & Sanity Check")
log("=" * 60)

datasets_raw = {
    "train":     load_jsonl(TRAIN_FILE),
    "dev":       load_jsonl(DEV_FILE),
    "train_neg": load_jsonl(TRAIN_NEG),
}

required_fields = ["query", "passage", "label", "meta"]
meta_fields     = ["van_ban", "dieu", "khoan", "chunk_index"]
summary_rows = []

for name, rows in datasets_raw.items():
    log(f"  Analyzing {name} ({len(rows)} rows) ...")
    pos = sum(1 for r in rows if r.get("label") == 1)
    neg = sum(1 for r in rows if r.get("label") == 0)
    missing_fields = sum(
        1 for r in rows if any(f not in r for f in required_fields)
        or any(f not in r.get("meta", {}) for f in meta_fields)
    )
    seen = {}
    conflicts = 0
    for r in rows:
        key = (r.get("query",""), r.get("passage",""))
        lbl = r.get("label")
        if key in seen and seen[key] != lbl: conflicts += 1
        seen[key] = lbl
    cjk_count   = sum(1 for r in rows if count_cjk(r.get("query","")) >= 3)
    pair_counts = Counter((r.get("query",""), r.get("passage","")) for r in rows)
    dup_pairs   = sum(1 for c in pair_counts.values() if c > 1)
    q_lens      = [len(r.get("query","")) for r in rows]
    p_lens      = [len(r.get("passage","")) for r in rows]
    summary_rows.append({
        "name": name, "rows": len(rows), "pos": pos, "neg": neg,
        "label_conflicts": conflicts, "cjk_queries": cjk_count,
        "dup_pairs": dup_pairs, "missing_fields": missing_fields,
        "avg_q_len": round(sum(q_lens)/len(q_lens),1) if q_lens else 0,
        "avg_p_len": round(sum(p_lens)/len(p_lens),1) if p_lens else 0,
    })
    log(f"    rows={len(rows)}, pos={pos}, neg={neg}, conflicts={conflicts}, cjk={cjk_count}")

# Save summary CSV
csv_path = EDA_DIR / "summary.csv"
fields = ["name","rows","pos","neg","label_conflicts","cjk_queries","dup_pairs","missing_fields","avg_q_len","avg_p_len"]
with open(csv_path, "w", newline="", encoding="utf-8") as f:
    w = csv.DictWriter(f, fieldnames=fields); w.writeheader(); w.writerows(summary_rows)

log(f"  Saved → {csv_path}")

# Plots
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

plot_dir = EDA_DIR / "plots"
fig, ax = plt.subplots(figsize=(8, 4))
x     = range(len(summary_rows))
width = 0.35
ax.bar([i-width/2 for i in x], [s["pos"] for s in summary_rows], width, label="Positive", color="#2196F3")
ax.bar([i+width/2 for i in x], [s["neg"] for s in summary_rows], width, label="Negative", color="#F44336")
ax.set_xticks(list(x)); ax.set_xticklabels([s["name"] for s in summary_rows])
ax.set_title("Label Distribution per Dataset"); ax.legend()
plt.tight_layout(); plt.savefig(plot_dir / "label_dist.png", dpi=120); plt.close()

all_rows = [r for rows in datasets_raw.values() for r in rows]
fig, ax  = plt.subplots(figsize=(8,4))
ax.hist([len(r.get("query","")) for r in all_rows], bins=50, color="#4CAF50", edgecolor="white")
ax.set_title("Query Length Distribution"); plt.tight_layout()
plt.savefig(plot_dir / "query_len.png", dpi=120); plt.close()

fig, ax = plt.subplots(figsize=(8,4))
ax.hist([len(r.get("passage","")) for r in all_rows], bins=50, color="#FF9800", edgecolor="white")
ax.set_title("Passage Length Distribution"); plt.tight_layout()
plt.savefig(plot_dir / "passage_len.png", dpi=120); plt.close()

vb_counts = Counter(r.get("meta",{}).get("van_ban","?") for r in all_rows)
top_vb    = vb_counts.most_common(20)
fig, ax   = plt.subplots(figsize=(10,5))
ax.barh([t[0] for t in top_vb],[t[1] for t in top_vb],color="#9C27B0"); ax.invert_yaxis()
ax.set_title("Top 20 Van Bản"); plt.tight_layout()
plt.savefig(plot_dir / "top_vanban.png", dpi=120); plt.close()

log("Phase A complete ✓")
print("\nSummary:")
for s in summary_rows:
    print(f"  {s['name']:12s}: {s['rows']} rows, pos={s['pos']}, neg={s['neg']}, conflicts={s['label_conflicts']}")

[03:06:23] ============================================================
[03:06:23] PHASE A: EDA & Sanity Check
[03:06:23] ============================================================
[03:06:23]   Analyzing train (6615 rows) ...
[03:06:23]     rows=6615, pos=6615, neg=0, conflicts=0, cjk=4
[03:06:23]   Analyzing dev (719 rows) ...
[03:06:23]     rows=719, pos=719, neg=0, conflicts=0, cjk=0
[03:06:23]   Analyzing train_neg (13230 rows) ...
[03:06:23]     rows=13230, pos=6615, neg=6615, conflicts=16, cjk=8
[03:06:23]   Saved → outputs\eda\summary.csv


C:\Users\ADMIN\AppData\Local\Temp\ipykernel_1900\2371801359.py:82: UserWarning: Tight layout not applied. The left and right margins cannot be made large enough to accommodate all Axes decorations.
  ax.set_title("Top 20 Van Bản"); plt.tight_layout()


[03:06:25] Phase A complete ✓

Summary:
  train       : 6615 rows, pos=6615, neg=0, conflicts=0
  dev         : 719 rows, pos=719, neg=0, conflicts=0
  train_neg   : 13230 rows, pos=6615, neg=6615, conflicts=16


---
## Phase B — Tạo `eval_qa.jsonl`

In [3]:
log("=" * 60)
log("PHASE B: Create eval_qa.jsonl")
log("=" * 60)

random.seed(SEED)
dev_rows       = load_jsonl(DEV_FILE)
train_neg_rows = load_jsonl(TRAIN_NEG)
vb_freq        = Counter(r.get("meta",{}).get("van_ban","") for r in train_neg_rows)

# Filter CJK
clean_dev   = [r for r in dev_rows if count_cjk(r.get("query","").strip()) < 3]
cjk_removed = len(dev_rows) - len(clean_dev)
log(f"  CJK filtered: {cjk_removed}, clean: {len(clean_dev)}")

# Group by (van_ban, dieu, khoan)
groups = {}
for r in clean_dev:
    meta = r.get("meta",{})
    key  = (meta.get("van_ban",""), meta.get("dieu",""), meta.get("khoan",""))
    groups.setdefault(key, []).append(r)

qa_items = []
for gid, (key, records) in enumerate(groups.items()):
    r    = random.choice(records)
    meta = r.get("meta",{})
    freq = vb_freq.get(meta.get("van_ban",""), 0)
    qa_items.append({
        "id": f"eval_{gid+1:04d}",
        "query": r.get("query","").strip(),
        "expected_citations": [{
            "van_ban":     meta.get("van_ban",""),
            "chuong":      meta.get("chuong",""),
            "dieu":        meta.get("dieu",""),
            "khoan":       meta.get("khoan",""),
            "diem":        meta.get("diem",""),
            "chunk_index": meta.get("chunk_index",-1),
        }],
        "tags":       ["easy" if freq >= 10 else "hard"],
        "source":     "dev.jsonl",
        "passage_ref": r.get("passage","")[:200],
    })

random.shuffle(qa_items)
write_jsonl(EVAL_QA_FILE, qa_items)
easy = sum(1 for q in qa_items if "easy" in q["tags"])
hard = sum(1 for q in qa_items if "hard" in q["tags"])
log(f"  Saved {len(qa_items)} QA items → {EVAL_QA_FILE} | easy={easy}, hard={hard}")
log("Phase B complete ✓")

[03:06:25] ============================================================
[03:06:25] PHASE B: Create eval_qa.jsonl
[03:06:25] ============================================================
[03:06:25]   CJK filtered: 0, clean: 719
[03:06:25]   Saved 323 QA items → outputs\eval\eval_qa.jsonl | easy=323, hard=0
[03:06:25] Phase B complete ✓


---
## Phase C — Train Cross-Encoder
Cell này mất ~10-30 phút tùy số epochs.

In [4]:
log("=" * 60)
log("PHASE C: Train Cross-Encoder")
log("=" * 60)

from sentence_transformers import CrossEncoder, InputExample
from sentence_transformers.cross_encoder.evaluation import CEBinaryClassificationEvaluator
from torch.utils.data import DataLoader

log(f"  Device: {DEVICE}")

# Build dev_with_neg.jsonl nếu chưa có
if not DEV_NEG_FILE.exists():
    log("  Building dev_with_neg.jsonl...")
    random.seed(SEED)
    neg_by_vb = {}
    for r in load_jsonl(TRAIN_NEG):
        if r.get("label") == 0:
            vb = r.get("meta",{}).get("van_ban","")
            neg_by_vb.setdefault(vb, []).append(r)
    dev_neg_rows = []
    for r in load_jsonl(DEV_FILE):
        meta = r.get("meta",{})
        vb, dieu = meta.get("van_ban",""), meta.get("dieu","")
        dev_neg_rows.append({"query": r["query"], "passage": r["passage"], "label": 1, "meta": meta})
        candidates = [n for n in neg_by_vb.get(vb,[]) if n.get("meta",{}).get("dieu","") != dieu]
        if not candidates:
            candidates = [n for n in load_jsonl(TRAIN_NEG) if n.get("label")==0 and n.get("meta",{}).get("van_ban","")!=vb]
        if candidates:
            neg = random.choice(candidates)
            dev_neg_rows.append({"query": r["query"], "passage": neg["passage"], "label": 0, "meta": neg.get("meta",{})})
    write_jsonl(DEV_NEG_FILE, dev_neg_rows)
    log(f"  Saved {len(dev_neg_rows)} rows → {DEV_NEG_FILE}")

# Prepare data
train_rows    = load_jsonl(TRAIN_NEG)
random.seed(SEED); random.shuffle(train_rows)
train_samples = [InputExample(texts=[r["query"],r["passage"]], label=float(r.get("label",0)))
                 for r in train_rows if "query" in r and "passage" in r and "label" in r]
dev_samples   = [InputExample(texts=[r["query"],r["passage"]], label=float(r.get("label",0)))
                 for r in load_jsonl(DEV_NEG_FILE)]
log(f"  Train: {len(train_samples)}, Dev: {len(dev_samples)}")

# Train
use_fp16 = (DEVICE == "cuda")
ce_model = CrossEncoder(BASE_CE_MODEL, num_labels=1, max_length=MAX_LENGTH, device=DEVICE)
MODEL_OUT.mkdir(parents=True, exist_ok=True)
evaluator = CEBinaryClassificationEvaluator.from_input_examples(dev_samples, name="dev")

t0 = time.time()
ce_model.fit(
    train_dataloader=DataLoader(train_samples, shuffle=True, batch_size=BATCH_SIZE),
    evaluator=evaluator,
    epochs=EPOCHS,
    warmup_steps=int(len(train_samples) / BATCH_SIZE * EPOCHS * 0.1),
    output_path=str(MODEL_OUT),
    use_amp=use_fp16,
)
elapsed = round((time.time()-t0)/60, 1)
log(f"  Training done in {elapsed} min")

saved_path = MODEL_OUT / "saved_model"
ce_model.save(str(saved_path))
log(f"  Model saved → {saved_path}")

config = {"base_model": BASE_CE_MODEL, "saved_path": str(saved_path),
          "epochs": EPOCHS, "batch_size": BATCH_SIZE, "max_length": MAX_LENGTH,
          "fp16": use_fp16, "device": DEVICE,
          "train_samples": len(train_samples), "training_minutes": elapsed}
(MODEL_OUT / "training_config.json").write_text(json.dumps(config, indent=2, ensure_ascii=False), encoding="utf-8")
log("Phase C complete ✓")

[03:06:25] ============================================================
[03:06:25] PHASE C: Train Cross-Encoder
[03:06:25] ============================================================


d:\SGU\CNTT\NCKH2025_2026\ChatBot\cross-encoder\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[03:06:34]   Device: cuda
[03:06:34]   Train: 13230, Dev: 1438


Loading weights: 100%|██████████| 105/105 [00:00<00:00, 723.63it/s, Materializing param=classifier.weight]                                    
BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Token indices sequence length is longer than the specified maximum sequence length for this model (280 > 256). Running this sequence through the model will result in indexing errors


Step,Training Loss
500,0.391855
1000,0.253884


[03:13:35]   Training done in 6.9 min


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  9.34it/s]

[03:13:35]   Model saved → outputs\models\cross_encoder_v1\saved_model
[03:13:35] Phase C complete ✓


---
## Phase D — Evaluate: Classification + Baseline + Reranking

In [5]:
log("=" * 60)
log("PHASE D: Evaluate Reranker")
log("=" * 60)

import faiss, numpy as np
from sentence_transformers import CrossEncoder, SentenceTransformer
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

# Load cross-encoder
ce_model_path = MODEL_OUT / "saved_model"
if not ce_model_path.exists(): ce_model_path = MODEL_OUT
ce_model = CrossEncoder(str(ce_model_path), max_length=MAX_LENGTH, device=DEVICE)
log(f"  Cross-encoder loaded from {ce_model_path}")

# D1: Classification
log("  [D1] Classification metrics...")
dev_neg_rows = load_jsonl(DEV_NEG_FILE)
y_true  = [int(r.get("label",0)) for r in dev_neg_rows]
scores  = ce_model.predict([(r["query"],r["passage"]) for r in dev_neg_rows], batch_size=BATCH_SIZE, show_progress_bar=True)
y_pred  = [1 if s >= 0.5 else 0 for s in scores]
cls_met = {
    "accuracy":  round(accuracy_score(y_true, y_pred), 4),
    "precision": round(precision_score(y_true, y_pred, zero_division=0), 4),
    "recall":    round(recall_score(y_true, y_pred, zero_division=0), 4),
    "f1":        round(f1_score(y_true, y_pred, zero_division=0), 4),
    "roc_auc":   round(roc_auc_score(y_true, scores), 4),
    "n_samples": len(y_true),
}
CLASS_METRICS.write_text(json.dumps(cls_met, indent=2, ensure_ascii=False), encoding="utf-8")
log(f"  Classification: {cls_met}")

# D2: Build FAISS
log("  [D2] Building FAISS index...")
bi_model = SentenceTransformer(BASE_BI_MODEL, device=DEVICE)
seen_passages = {}
for f in [TRAIN_FILE, DEV_FILE, TRAIN_NEG]:
    for r in load_jsonl(f):
        p = r.get("passage","")
        if p and p not in seen_passages:
            meta = r.get("meta",{})
            seen_passages[p] = {"passage":p,
                "chunk_index":meta.get("chunk_index",-1), "van_ban":meta.get("van_ban",""),
                "chuong":meta.get("chuong",""), "dieu":meta.get("dieu",""),
                "khoan":meta.get("khoan",""), "diem":meta.get("diem","")}
corpus     = list(seen_passages.values())
embeddings = bi_model.encode([c["passage"] for c in corpus], batch_size=64,
                              show_progress_bar=True, convert_to_numpy=True,
                              normalize_embeddings=True).astype("float32")
index_d = faiss.IndexFlatIP(embeddings.shape[1])
index_d.add(embeddings)
faiss.write_index(index_d, str(FAISS_INDEX))
mapping_d = [{"faiss_id":i,**c} for i,c in enumerate(corpus)]
write_jsonl(FAISS_MAP, mapping_d)
log(f"  FAISS: {len(corpus)} passages | index → {FAISS_INDEX}")

# D3: Ranking metrics
log("  [D3] Ranking metrics...")
eval_qa = load_jsonl(EVAL_QA_FILE)

def is_hit_d(faiss_id, ec, m):
    row = m[faiss_id]
    for e in ec:
        ci = e.get("chunk_index",-2)
        if ci != -1 and row["chunk_index"] == ci: return True
        if row["van_ban"]==e.get("van_ban","") and row["dieu"]==e.get("dieu","") and row["khoan"]==e.get("khoan",""): return True
    return False

r_base  = {"R@1":[],"R@3":[],"R@5":[],"MRR@10":[]}
r_rerank= {"R@1":[],"R@3":[],"R@5":[],"MRR@10":[]}

from tqdm import tqdm
for item in tqdm(eval_qa, desc="Phase D eval"):
    query = item["query"]; ec = item["expected_citations"]
    qe    = bi_model.encode([query], normalize_embeddings=True, convert_to_numpy=True).astype("float32")
    _, ids  = index_d.search(qe, TOP_N)
    ids     = ids[0].tolist()

    for k, key in [(1,"R@1"),(3,"R@3"),(5,"R@5")]:
        r_base[key].append(1 if any(is_hit_d(i,ec,mapping_d) for i in ids[:k] if i>=0) else 0)
    mrr = 0.0
    for rank,i in enumerate(ids[:10],1):
        if i>=0 and is_hit_d(i,ec,mapping_d): mrr=1.0/rank; break
    r_base["MRR@10"].append(mrr)

    cands   = [(mapping_d[i]["passage"],i) for i in ids if i>=0]
    rscores = ce_model.predict([[query,c[0]] for c in cands], batch_size=BATCH_SIZE) if cands else []
    ranked  = sorted(zip(rscores,[c[1] for c in cands]),reverse=True)
    r_ids   = [r[1] for r in ranked]

    for k, key in [(1,"R@1"),(3,"R@3"),(5,"R@5")]:
        r_rerank[key].append(1 if any(is_hit_d(i,ec,mapping_d) for i in r_ids[:k]) else 0)
    mrr = 0.0
    for rank,i in enumerate(r_ids[:10],1):
        if is_hit_d(i,ec,mapping_d): mrr=1.0/rank; break
    r_rerank["MRR@10"].append(mrr)

rerank_rows = [
    {"metric":"Recall@1",  "baseline":avg(r_base["R@1"]),   "reranked":avg(r_rerank["R@1"])},
    {"metric":"Recall@3",  "baseline":avg(r_base["R@3"]),   "reranked":avg(r_rerank["R@3"])},
    {"metric":"Recall@5",  "baseline":avg(r_base["R@5"]),   "reranked":avg(r_rerank["R@5"])},
    {"metric":"MRR@10",    "baseline":avg(r_base["MRR@10"]),"reranked":avg(r_rerank["MRR@10"])},
]
with open(RERANK_CSV, "w", newline="", encoding="utf-8") as f:
    w = csv.DictWriter(f, fieldnames=["metric","baseline","reranked"])
    w.writeheader(); w.writerows(rerank_rows)

print("\n── Phase D Results ──")
print(f"  {'Metric':<12} {'Baseline':>10} {'Reranked':>10} {'Δ':>8}")
for row in rerank_rows:
    delta = float(row['reranked']) - float(row['baseline'])
    print(f"  {row['metric']:<12} {row['baseline']:>10} {row['reranked']:>10} {delta:>+8.4f}")
log(f"  Saved → {RERANK_CSV}")
log("Phase D complete ✓")

[03:13:35] ============================================================
[03:13:35] PHASE D: Evaluate Reranker
[03:13:35] ============================================================


Loading weights: 100%|██████████| 105/105 [00:00<00:00, 785.42it/s, Materializing param=classifier.weight]                                    


[03:13:36]   Cross-encoder loaded from outputs\models\cross_encoder_v1\saved_model
[03:13:36]   [D1] Classification metrics...


Batches: 100%|██████████| 45/45 [00:08<00:00,  5.54it/s]


[03:13:44]   Classification: {'accuracy': 0.8797, 'precision': 0.8447, 'recall': 0.9305, 'f1': 0.8855, 'roc_auc': 0.9564, 'n_samples': 1438}
[03:13:44]   [D2] Building FAISS index...


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 758.76it/s, Materializing param=pooler.dense.weight]                               
BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Batches: 100%|██████████| 30/30 [00:09<00:00,  3.29it/s]

[03:14:02]   FAISS: 1861 passages | index → outputs\tmp\faiss.index


[03:14:02]   [D3] Ranking metrics...


Phase D eval: 100%|██████████| 323/323 [01:47<00:00,  3.00it/s]


── Phase D Results ──
  Metric         Baseline   Reranked        Δ
  Recall@1         0.2074     0.4458  +0.2384
  Recall@3         0.3529     0.5511  +0.1982
  Recall@5         0.4087     0.5789  +0.1702
  MRR@10           0.2908     0.5036  +0.2128
[03:15:49]   Saved → outputs\eval\rerank_metrics.csv
[03:15:49] Phase D complete ✓


---
## Phase E — End-to-End Pipeline
> Retrieve → Rerank → Gate → (LLM nếu có API key)

In [6]:
log("=" * 60)
log("PHASE E: Pipeline End-to-End")
log("=" * 60)

# Dùng FAISS + models từ Phase D (đã load)
# Nếu chạy độc lập, uncomment dòng dưới:
# ce_model = CrossEncoder(str(MODEL_OUT/"saved_model"), max_length=MAX_LENGTH, device=DEVICE)
# bi_model = SentenceTransformer(BASE_BI_MODEL, device=DEVICE)
# index_d  = faiss.read_index(str(FAISS_INDEX))
# mapping_d = load_jsonl(FAISS_MAP)

GATE_PASS   = 0.5
GATE_REFINE = 0.3

# LLM client (placeholder nếu không có API key)
class LLMClient:
    def __init__(self):
        self.mode = "placeholder"
        if os.environ.get("OPENROUTER_API_KEY"): self.mode = "openrouter"
        elif os.environ.get("GEMINI_API_KEY"): self.mode = "gemini"
        log(f"  LLM mode: {self.mode}")
    def generate(self, query, passages):
        return None  # placeholder

llm = LLMClient()

eval_qa_e = load_jsonl(EVAL_QA_FILE)
random.seed(SEED); random.shuffle(eval_qa_e)
eval_qa_e = eval_qa_e[:EVAL_N]
log(f"  Evaluating {len(eval_qa_e)} questions")

def is_hit_e(faiss_id, ec):
    row = mapping_d[faiss_id]
    for e in ec:
        ci = e.get("chunk_index",-2)
        if ci != -1 and row.get("chunk_index")==ci: return True
        if row.get("van_ban")==e.get("van_ban") and row.get("dieu")==e.get("dieu") and row.get("khoan")==e.get("khoan"): return True
    return False

results_e = []
for item in tqdm(eval_qa_e, desc="Phase E"):
    t0 = time.time()
    query = item["query"]; ec = item["expected_citations"]
    qe    = bi_model.encode([query], normalize_embeddings=True, convert_to_numpy=True).astype("float32")
    sf, ids_raw = index_d.search(qe, TOP_N)
    ids    = [i for i in ids_raw[0].tolist() if i >= 0]
    sfs    = sf[0].tolist()

    cands   = [(mapping_d[i]["passage"],i,sf_) for i,sf_ in zip(ids,sfs) if i<len(mapping_d)]
    rscores = ce_model.predict([[query,c[0]] for c in cands], batch_size=BATCH_SIZE) if cands else []
    ranked  = sorted(zip(rscores,cands),reverse=True) if len(rscores)>0 else []
    reranked_ids    = [c[1] for _,c in ranked]
    reranked_scores = [float(s) for s,_ in ranked]

    max_score = reranked_scores[0] if reranked_scores else 0.0
    decision  = "PASS" if max_score>=GATE_PASS else ("REFINE" if max_score>=GATE_REFINE else "ABSTAIN")

    top3_ids     = reranked_ids[:3]
    citation_hit = any(is_hit_e(i,ec) for i in top3_ids if i<len(mapping_d))
    answer       = "Tôi không tìm thấy thông tin phù hợp." if decision=="ABSTAIN" else None

    results_e.append({
        "id": item["id"], "query": query,
        "expected_citations": ec,
        "reranked_ids": reranked_ids[:10],
        "reranked_scores": [round(s,4) for s in reranked_scores[:10]],
        "decision": decision,
        "citation_hit": citation_hit,
        "answer": answer,
        "latency_ms": round((time.time()-t0)*1000,1),
    })

write_jsonl(PIPE_RESULTS, results_e)
n = len(results_e)
summary_e = [
    {"metric":"total_questions",   "value": n},
    {"metric":"citation_hit_rate", "value": round(sum(1 for r in results_e if r["citation_hit"])/n,4)},
    {"metric":"abstain_rate",       "value": round(sum(1 for r in results_e if r["decision"]=="ABSTAIN")/n,4)},
    {"metric":"pass_rate",          "value": round(sum(1 for r in results_e if r["decision"]=="PASS")/n,4)},
    {"metric":"avg_latency_ms",     "value": round(sum(r["latency_ms"] for r in results_e)/n,1)},
]
with open(PIPE_SUM_CSV,"w",newline="",encoding="utf-8") as f:
    w = csv.DictWriter(f,fieldnames=["metric","value"]); w.writeheader(); w.writerows(summary_e)

print("\n── Phase E Results ──")
for s in summary_e: print(f"  {s['metric']}: {s['value']}")
log(f"  Saved → {PIPE_RESULTS}, {PIPE_SUM_CSV}")
log("Phase E complete ✓")

[03:15:49] ============================================================
[03:15:49] PHASE E: Pipeline End-to-End
[03:15:49] ============================================================
[03:15:49]   LLM mode: placeholder
[03:15:49]   Evaluating 50 questions


Phase E: 100%|██████████| 50/50 [00:17<00:00,  2.93it/s]


── Phase E Results ──
  total_questions: 50
  citation_hit_rate: 0.64
  abstain_rate: 0.02
  pass_rate: 0.98
  avg_latency_ms: 340.8
[03:16:06]   Saved → outputs\eval\pipeline_results.jsonl, outputs\eval\pipeline_summary.csv
[03:16:06] Phase E complete ✓


---
## Phase F — Tạo DELIVERABLES.md

In [7]:
log("=" * 60)
log("PHASE F: DELIVERABLES.md")
log("=" * 60)

rerank_data = []
if RERANK_CSV.exists():
    with open(RERANK_CSV, encoding="utf-8") as f:
        rerank_data = list(csv.DictReader(f))
pipe_sum = []
if PIPE_SUM_CSV.exists():
    with open(PIPE_SUM_CSV, encoding="utf-8") as f:
        pipe_sum = list(csv.DictReader(f))
met_d = json.loads(CLASS_METRICS.read_text(encoding="utf-8")) if CLASS_METRICS.exists() else {}

with open(DELIVERABLES, "w", encoding="utf-8") as f:
    f.write("# Deliverables — Cross-Encoder Reranker Pipeline\n\n")
    f.write(f"Generated: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n\n")
    f.write(f"## Environment\n- Python: {sys.version.split()[0]}  \n- GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'N/A'}\n\n")

    if met_d:
        f.write("## Phase D — Classification Metrics\n\n| Metric | Value |\n|--------|-------|\n")
        for k,v in met_d.items(): f.write(f"| {k} | {v} |\n")
        f.write("\n")

    if rerank_data:
        f.write("## Phase D — Ranking Metrics\n\n| Metric | Baseline | Reranked |\n|--------|----------|----------|\n")
        for row in rerank_data: f.write(f"| {row['metric']} | {row['baseline']} | {row['reranked']} |\n")
        f.write("\n")

    if pipe_sum:
        f.write("## Phase E — Pipeline Metrics\n\n| Metric | Value |\n|--------|-------|\n")
        for row in pipe_sum: f.write(f"| {row['metric']} | {row['value']} |\n")
        f.write("\n")

    f.write("## Reproduction\n\n```bash\n")
    f.write("# Chạy notebook: run_pipeline.ipynb (Run All)\n")
    f.write("# Hoặc script:   ./venv/python.exe run_pipeline.py --phase ALL --seed 42\n")
    f.write("```\n")

log(f"  Saved → {DELIVERABLES}")
log("Phase F complete ✓")
log("=" * 60)
log("ALL PHASES COMPLETE ✓")

[03:16:06] ============================================================
[03:16:06] PHASE F: DELIVERABLES.md
[03:16:06] ============================================================
[03:16:06]   Saved → outputs\DELIVERABLES.md
[03:16:06] Phase F complete ✓
[03:16:06] ============================================================
[03:16:06] ALL PHASES COMPLETE ✓
